# Materialized view

This page considers the specifics of working with materialized views in ClickHouse.

## Engine

In the line with the concept of materialized views, ClickHouse engines can be configured for them, since they are still tables that physically store the data.

Specy the `ENGINE=` clause after the name of the view.

---

Consider the following practical example: some information was randomly appended to the source data, and the materialization view has to eliminate the duplicate data.

The following cell creates a source table and MV with `ReplacingMergeTree` refering to the `data_source`:

In [40]:
--ClickHouse
DROP TABLE IF EXISTS data_source;
DROP TABLE IF EXISTS target_category;

CREATE TABLE data_source
(
    vals Array(Int8)
);

CREATE MATERIALIZED
VIEW target_category
ENGINE=ReplacingMergeTree 
ORDER BY vals AS
SELECT vals
FROM data_source
ARRAY JOIN data_source.vals AS vals;

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,1010616,1050807,cacda96d-6187-43ed-a135-ae18c6ea2e6d


read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,5962834,1113639,c669944d-c11f-4f5d-be64-12d2a9c1e4c8


read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,6310764,1216071,e776912e-801a-4a92-b8a4-32260cb97a13


The insertion to the data source indicates that the MV would finally have unique values.

In [41]:
--ClickHouse
INSERT INTO data_source VALUES
([10, 20, 30, 20]);
SELECT * FROM target_category FINAL;

vals
10
20
30


The another append to the data source still ensures that the content of the MV renains unique.

In [42]:
--ClickHouse
INSERT INTO data_source VALUES
([20, 40]);
SELECT * FROM target_category FINAL;

vals
10
30
20
40
